# Unify AESO history (CSV) + live API into normalized targets

For each concept we build one **staging** streaming table fed by two append flows
— the historical wide CSV (melted to long) and the bronze API table's Change Data
Feed — then run **two** AUTO CDC flows off that staging table:

* a **current** SCD type 1 table (latest-wins), and
* a **history** SCD type 2 table (`*_hist`, keeps `__START_AT`/`__END_AT`).

| concept | current / history | keys |
|---|---|---|
| generation | `generation` / `generation_hist` | `begin_datetime_utc, asset_id` |
| pool price | `pool_price` / `pool_price_hist` | `begin_datetime_utc` |
| AIL | `ail` / `ail_hist` | `begin_datetime_utc` |
| interchange | `interchange` / `interchange_hist` | `begin_datetime_utc, path` |

**Sequencing:** every staging row carries `_seq`. CSV rows are `0`; API rows are
`_commit_version + 1`, so the API always wins at the seam and later settlement
revisions (later CDF commits) win over earlier ones. The historical seam is clean
(CSV ends 2025-07-31 23:00 MPT, API starts 2025-08-01 00:00 MPT) so the two
sources cover disjoint hours and never actually collide on a key.

The old wide `pool_price_ail_historical` table is intentionally **dropped** from
this pipeline; these normalized targets replace it.


In [ ]:
from pyspark import pipelines as dp
from pyspark.sql import functions as F

catalog_name = spark.conf.get("source_catalog")
schema_name = spark.conf.get("source_schema")
volume_name = spark.conf.get("source_volume")

volume_path = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}/volume_pool-price_ail_historical/"

# CSV timestamp parsing. The three source files disagree on date format: the
# 2001-2009 and 2020-Jul2025 files use yyyy-MM-dd, but 2010-2019 uses M/d/yyyy.
# Coalesce both (single-digit hour), parsed naively to the same wall clock the bronze
# API tables use, so the Jul/Aug 2025 seam lines up. A wrong/missing format would null
# the timestamp and silently drop those rows (null key) in AUTO CDC.
CSV_TS = (
    "coalesce("
    "to_timestamp(Date_Begin_GMT, 'yyyy-MM-dd H:mm'), "
    "to_timestamp(Date_Begin_GMT, 'M/d/yyyy H:mm')"
    ") as begin_datetime_utc"
)

# everything in the wide CSV that is NOT a per-asset generation column
NON_GEN_COLS = {
    "Date_Begin_GMT",
    "Date_Begin_Local",
    "ACTUAL_POOL_PRICE",
    "HOUR_AHEAD_POOL_PRICE_FORECAST",
    "ACTUAL_AIL",
    "EXPORT_BC",
    "EXPORT_SK",
    "IMPORT_BC",
    "IMPORT_SK",
    "_rescued_data",
}


def csv_raw():
    """Fresh Auto Loader stream over the historical CSV drops."""
    return (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .load(volume_path)
    )


def bronze_cdf(table):
    """Stream the bronze table's change feed, keeping only live (non-preimage) rows."""
    return (
        spark.readStream.option("readChangeFeed", "true")
        .table(f"{catalog_name}.{schema_name}.{table}")
        .filter(F.col("_change_type").isin("insert", "update_postimage"))
    )


def api_seq():
    """CDF commit order as the sequence value; always > the CSV's 0."""
    return (F.col("_commit_version") + F.lit(1)).cast("long")


def land(concept, keys, comment):
    """Two AUTO CDC flows off stg_<concept>: SCD1 current + SCD2 history."""
    dp.create_streaming_table(
        name=concept, comment=f"{comment} Current value per key (SCD type 1)."
    )
    dp.create_auto_cdc_flow(
        target=concept,
        source=f"stg_{concept}",
        keys=keys,
        sequence_by="_seq",
        stored_as_scd_type=1,
        except_column_list=["_seq"],
    )
    dp.create_streaming_table(
        name=f"{concept}_hist",
        comment=f"{comment} Full revision history with __START_AT/__END_AT (SCD type 2); shows preliminary-vs-settled values.",
    )
    dp.create_auto_cdc_flow(
        target=f"{concept}_hist",
        source=f"stg_{concept}",
        keys=keys,
        sequence_by="_seq",
        stored_as_scd_type=2,
        except_column_list=["_seq"],
    )

## Generation

Melt the wide per-asset CSV columns to `(asset_id, generation_mw)` via `stack` (streaming-safe; null cells from the merged multi-file schema are dropped), unioned through staging with the bronze generation CDF.

In [ ]:
dp.create_streaming_table(
    name="stg_generation",
    comment="Internal staging: union of historical CSV + bronze API change feed, consumed by the AUTO CDC flows.",
)


@dp.append_flow(target="stg_generation", name="generation_csv")
def generation_csv():
    df = csv_raw()
    gen_cols = [c for c in df.columns if c not in NON_GEN_COLS]
    pairs = ", ".join(f"'{c}', try_cast(`{c}` AS double)" for c in gen_cols)
    stack_expr = f"stack({len(gen_cols)}, {pairs}) as (asset_id, generation_mw)"
    return (
        df.selectExpr(CSV_TS, stack_expr)
        .filter(
            F.col("generation_mw").isNotNull()
        )  # drop assets absent from a given CSV file
        .select(
            "begin_datetime_utc",
            "asset_id",
            "generation_mw",
            F.lit("historical_csv").alias("source"),
            F.lit(0).cast("long").alias("_seq"),
        )
    )


@dp.append_flow(target="stg_generation", name="generation_api")
def generation_api():
    return bronze_cdf("bronze_generation").select(
        "begin_datetime_utc",
        "asset_id",
        F.col("metered_volume").cast("double").alias("generation_mw"),
        F.lit("aeso_api").alias("source"),
        api_seq().alias("_seq"),
    )


land(
    "generation",
    ["begin_datetime_utc", "asset_id"],
    "AESO hourly net metered generation (MWh) per asset, unifying the historical CSV with the live Metered Volume API.",
)

## Pool price

In [ ]:
dp.create_streaming_table(
    name="stg_pool_price",
    comment="Internal staging: union of historical CSV + bronze API change feed, consumed by the AUTO CDC flows.",
)


@dp.append_flow(target="stg_pool_price", name="pool_price_csv")
def pool_price_csv():
    return csv_raw().select(
        F.expr(CSV_TS),
        F.expr("try_cast(ACTUAL_POOL_PRICE as double)").alias("pool_price"),
        F.expr("try_cast(HOUR_AHEAD_POOL_PRICE_FORECAST as double)").alias(
            "forecast_pool_price"
        ),
        F.lit(None).cast("double").alias("rolling_30day_avg"),  # not in the CSV
        F.lit("historical_csv").alias("source"),
        F.lit(0).cast("long").alias("_seq"),
    )


@dp.append_flow(target="stg_pool_price", name="pool_price_api")
def pool_price_api():
    return bronze_cdf("bronze_pool_price").select(
        "begin_datetime_utc",
        "pool_price",
        "forecast_pool_price",
        "rolling_30day_avg",
        F.lit("aeso_api").alias("source"),
        api_seq().alias("_seq"),
    )


land(
    "pool_price",
    ["begin_datetime_utc"],
    "AESO hourly pool price (CAD/MWh): actual, hour-ahead forecast, and (API era) rolling 30-day average.",
)

## Alberta Internal Load (AIL)

In [ ]:
dp.create_streaming_table(
    name="stg_ail",
    comment="Internal staging: union of historical CSV + bronze API change feed, consumed by the AUTO CDC flows.",
)


@dp.append_flow(target="stg_ail", name="ail_csv")
def ail_csv():
    return csv_raw().select(
        F.expr(CSV_TS),
        F.expr("try_cast(ACTUAL_AIL as double)").alias("alberta_internal_load"),
        F.lit(None)
        .cast("double")
        .alias("forecast_alberta_internal_load"),  # not in the CSV
        F.lit("historical_csv").alias("source"),
        F.lit(0).cast("long").alias("_seq"),
    )


@dp.append_flow(target="stg_ail", name="ail_api")
def ail_api():
    return bronze_cdf("bronze_ail").select(
        "begin_datetime_utc",
        "alberta_internal_load",
        "forecast_alberta_internal_load",
        F.lit("aeso_api").alias("source"),
        api_seq().alias("_seq"),
    )


land(
    "ail",
    ["begin_datetime_utc"],
    "AESO hourly Alberta Internal Load (MW): actual and (API era) forecast.",
)

## Interchange (interties)

Melt the four CSV intertie columns to `(path, import_mw, export_mw)` for BC and SK via `stack`; the bronze API side adds Montana (MT).

In [ ]:
dp.create_streaming_table(
    name="stg_interchange",
    comment="Internal staging: union of historical CSV + bronze API change feed, consumed by the AUTO CDC flows.",
)


@dp.append_flow(target="stg_interchange", name="interchange_csv")
def interchange_csv():
    stack_expr = (
        "stack(2, "
        "'BC', try_cast(IMPORT_BC as double), try_cast(EXPORT_BC as double), "
        "'SK', try_cast(IMPORT_SK as double), try_cast(EXPORT_SK as double)) "
        "as (path, import_mw, export_mw)"
    )
    return (
        csv_raw()
        .selectExpr(CSV_TS, stack_expr)
        .withColumn("net_export_mw", F.col("export_mw") - F.col("import_mw"))
        .select(
            "begin_datetime_utc",
            "path",
            "import_mw",
            "export_mw",
            "net_export_mw",
            F.lit("historical_csv").alias("source"),
            F.lit(0).cast("long").alias("_seq"),
        )
    )


@dp.append_flow(target="stg_interchange", name="interchange_api")
def interchange_api():
    return bronze_cdf("bronze_interchange").select(
        "begin_datetime_utc",
        "path",
        "import_mw",
        "export_mw",
        "net_export_mw",
        F.lit("aeso_api").alias("source"),
        api_seq().alias("_seq"),
    )


land(
    "interchange",
    ["begin_datetime_utc", "path"],
    "AESO hourly intertie flows by path (BC, SK, and — API era — MT): import, export, and net export (MWh).",
)